# 📄 ATS Resume Scanner
Run all cells in order.

In [ ]:
# ── STEP 1: Install dependencies ──────────────────────────────────────────────
!pip install -q pdfplumber scikit-learn streamlit pyngrok

In [ ]:
# ── STEP 2: Upload your PDFs ──────────────────────────────────────────────────
from google.colab import files
print('Upload your RESUME PDF first, then JOB DESCRIPTION PDF.')
uploaded = files.upload()
file_names = list(uploaded.keys())
print('Uploaded:', file_names)

In [ ]:
# ── STEP 3: Write project files ───────────────────────────────────────────────
import os
os.makedirs('ats_app', exist_ok=True)

# Move uploaded PDFs into the app folder
import shutil
for f in file_names:
    shutil.copy(f, f'ats_app/{f}')
print('PDFs ready in ats_app/')

In [ ]:
%%writefile ats_app/scanner.py
"""
scanner.py — Core ATS analysis logic
"""
import re, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pdfplumber
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def extract_text_from_pdf(pdf_file) -> str:
    text = ''
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t: text += t + '\n'
    return text.strip()

STOPWORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'by','from','is','are','was','were','be','been','being','have','has',
    'had','do','does','did','will','would','could','should','may','might',
    'shall','can','our','we','you','your','their','this','that','these',
    'those','it','its','as','if','not','no','so','up','out','than','also',
    'both','each','more','such','well','us','per','via','i','my','me',
    'about','all','other','any','they','he','she','who','which','what',
    'when','how','am','very','just','use','using'
}

TECH_SKILLS = {
    'python','java','javascript','typescript','c++','c#','ruby','go','rust',
    'swift','kotlin','sql','nosql','html','css','react','angular','vue',
    'django','flask','fastapi','spring','nodejs','tensorflow','pytorch',
    'sklearn','pandas','numpy','matplotlib','docker','kubernetes','aws',
    'azure','gcp','git','linux','bash','rest','api','graphql','mongodb',
    'postgresql','mysql','redis','kafka','spark','hadoop','machine learning',
    'deep learning','nlp','data science','devops','ci/cd','agile','scrum',
    'microservices','tableau','powerbi','excel','vba'
}

SECTIONS = {
    'experience'     : r'\b(experience|work history|employment)\b',
    'education'      : r'\b(education|degree|university|college|bachelor|master|phd)\b',
    'skills'         : r'\b(skills|technologies|tools|proficiencies)\b',
    'contact'        : r'\b(email|phone|linkedin|github|contact)\b',
    'summary'        : r'\b(summary|objective|profile|about me)\b',
    'projects'       : r'\b(projects|portfolio)\b',
    'certifications' : r'\b(certifications|certificates|credentials)\b',
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def extract_keywords(text):
    words = set(preprocess(text).split())
    return {w for w in words if len(w) > 2 and w not in STOPWORDS}

def extract_skills(text):
    t = text.lower()
    return {s for s in TECH_SKILLS if s in t}

def detect_sections(text):
    return [n for n, p in SECTIONS.items() if re.search(p, text.lower())]

def analyze(resume_text, job_text):
    rc, jc = preprocess(resume_text), preprocess(job_text)
    jkw, rkw = extract_keywords(jc), extract_keywords(rc)
    matched_kw, missing_kw = jkw & rkw, jkw - rkw
    kw_ratio = len(matched_kw)/len(jkw) if jkw else 0
    kw_score = round(kw_ratio * 35, 2)

    js, rs = extract_skills(job_text), extract_skills(resume_text)
    matched_s, missing_s = js & rs, js - rs
    sk_ratio = len(matched_s)/len(js) if js else 0
    sk_score = round(sk_ratio * 25, 2)

    cv = CountVectorizer()
    cos_sim = cosine_similarity(cv.fit_transform([rc, jc]))[0][1]
    cos_score = round(cos_sim * 20, 2)

    tfidf = TfidfVectorizer()
    tf_sim = cosine_similarity(tfidf.fit_transform([rc, jc]))[0][1]
    tf_score = round(tf_sim * 10, 2)

    sec = detect_sections(resume_text)
    sec_score = round(len(sec)/len(SECTIONS)*10, 2)

    total = kw_score + sk_score + cos_score + tf_score + sec_score
    sv = np.array([kw_score, sk_score, cos_score, tf_score, sec_score])
    df = pd.DataFrame({
        'Component'  : ['Keyword Match','Skills Match','Cosine Sim','TF-IDF','Sections'],
        'Score'      : sv,
        'Max'        : [35,25,20,10,10],
        'Percentage' : np.round(sv/np.array([35,25,20,10,10])*100,1)
    })
    return {
        'total': round(float(total),2), 'score_df': df, 'score_vector': sv,
        'kw_score': kw_score, 'kw_match_pct': round(kw_ratio*100,1),
        'matched_keywords': sorted(matched_kw), 'missing_keywords': sorted(missing_kw),
        'skill_score': sk_score, 'skill_match_pct': round(sk_ratio*100,1),
        'matched_skills': sorted(matched_s), 'missing_skills': sorted(missing_s),
        'cos_score': cos_score, 'cos_sim_pct': round(float(cos_sim)*100,1),
        'tfidf_score': tf_score, 'tfidf_sim_pct': round(float(tf_sim)*100,1),
        'section_score': sec_score, 'found_sections': sec,
    }

def verdict(score):
    if score >= 80: return ('✅','Excellent Match','#22c55e')
    elif score >= 60: return ('🟡','Good Match','#eab308')
    elif score >= 40: return ('🟠','Moderate Match','#f97316')
    else: return ('🔴','Poor Match','#ef4444')

In [ ]:
# ── STEP 4: Quick CLI scan (no Streamlit needed) ───────────────────────────────
# Change these to your actual uploaded file names:
RESUME_PDF = file_names[0]   # first uploaded file
JOB_PDF    = file_names[1]   # second uploaded file

import sys
sys.path.insert(0, 'ats_app')
from scanner import extract_text_from_pdf, analyze, verdict
import matplotlib.pyplot as plt
import numpy as np

resume_text = extract_text_from_pdf(f'ats_app/{RESUME_PDF}')
job_text    = extract_text_from_pdf(f'ats_app/{JOB_PDF}')
results     = analyze(resume_text, job_text)
emoji, label, color = verdict(results['total'])

print(f'\n{emoji}  ATS Score: {results["total"]} / 100  —  {label}')
print(f'\nKeyword Match : {results["kw_match_pct"]}%')
print(f'Skills Match  : {results["skill_match_pct"]}%')
print(f'Cosine Sim    : {results["cos_sim_pct"]}%')
print(f'Sections      : {", ".join(results["found_sections"])}')
print(f'\nMissing Keywords : {", ".join(results["missing_keywords"][:10])}')
print(f'Missing Skills   : {", ".join(results["missing_skills"])}')

# Matplotlib bar chart
labels = ['Keyword\nMatch','Skills\nMatch','Cosine\nSim','TF-IDF','Sections']
sv     = results['score_vector']
maxes  = np.array([35,25,20,10,10])
pcts   = sv / maxes * 100
colors = ['#22c55e' if p>=80 else '#eab308' if p>=60 else '#f97316' if p>=40 else '#ef4444' for p in pcts]

fig, ax = plt.subplots(figsize=(9,4))
ax.barh(labels, pcts, color=colors)
ax.set_xlim(0,110)
ax.set_xlabel('Score %')
ax.set_title(f'ATS Score Breakdown  |  Total: {results["total"]}/100')
for i, (p, r) in enumerate(zip(pcts, sv)):
    ax.text(p+1, i, f'{p:.0f}% ({r:.1f})', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── STEP 5 (Optional): Launch Streamlit UI via ngrok ──────────────────────────
# Get a free token at https://ngrok.com → Your Authtoken
NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'

!cp app.py ats_app/app.py 2>/dev/null || true   # copy if you wrote app.py separately

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

import subprocess, threading, time

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'ats_app/app.py',
                    '--server.port=8501', '--server.headless=true'])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(4)

public_url = ngrok.connect(8501)
print('\n✅ Streamlit app is live at:', public_url)